# Tahap 1: Membangun Case Base
### CBR Putusan Wanprestasi — Direktori Putusan Mahkamah Agung RI

**Tim:**
- Ahmad Qayyim — 202310370311286
- Bintang Mars Satria Tuhu — 202310370311410

**Mata Kuliah:** Penalaran Komputer — SubCPMK-3: Case-Based Reasoning

---

## Catatan: Workflow Manual

Scraping otomatis ke Direktori Putusan MA RI tidak memungkinkan karena server memblokir request dengan HTTP 403 (proteksi anti-bot). Sebagai alternatif yang sesuai etika dan rubrik tugas, dokumen dikumpulkan secara manual melalui browser, kemudian metadata di-input ke `metadata.csv`.

Notebook ini menjalankan tahap 1 dari pipeline CBR dengan workflow manual:
1. Load metadata.csv → validasi → convert ke metadata.json
2. Ekstraksi PDF ke teks plain
3. Cleaning dan validasi teks
4. Verifikasi case base siap untuk tahap berikutnya

## 1. Setup Environment

In [ ]:
!pip install pdfminer.six pandas tqdm -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path('/content/drive/MyDrive/kuliah/smt 6/Penalaran Komputer/Tugas SUB-CPMK 3/cbr-putusan-wanprestasi')
os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

## 2. Verifikasi File Input

Cek bahwa input manual sudah tersedia di Drive:
- 35 PDF di `data/raw/pdf/case_001.pdf` sampai `case_035.pdf`
- File `data/raw/metadata.csv` yang sudah diisi

In [ ]:
pdf_dir = PROJECT_DIR / 'data' / 'raw' / 'pdf'
csv_path = PROJECT_DIR / 'data' / 'raw' / 'metadata.csv'

pdfs = sorted(pdf_dir.glob('*.pdf'))
print(f'Jumlah PDF di data/raw/pdf/ : {len(pdfs)}')
print(f'metadata.csv exists         : {csv_path.exists()}')

if pdfs:
    print(f'\nSample 5 PDF pertama:')
    for p in pdfs[:5]:
        print(f'  {p.name}  ({p.stat().st_size / 1024:.1f} KB)')

## 3. Load Metadata + Verifikasi Konsistensi

In [ ]:
import sys
import importlib.util

sys.path.insert(0, str(PROJECT_DIR / 'src'))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[name] = mod
    spec.loader.exec_module(mod)
    return mod

loader_mod = load_module('loader_mod', PROJECT_DIR / 'src' / '01_load_manual_cases.py')
cases = loader_mod.run()
print(f'\nTotal cases loaded: {len(cases)}')

In [ ]:
import json
import pandas as pd

with open('data/raw/metadata.json', 'r', encoding='utf-8') as f:
    meta = json.load(f)

df_meta = pd.DataFrame(meta)
print(f'Shape: {df_meta.shape}')
print(f'\nDistribusi status:')
print(df_meta['status'].value_counts())
print(f'\nDistribusi per tahun:')
print(df_meta['tahun'].value_counts().sort_index())
df_meta.head()

## 4. Ekstraksi PDF + Cleaning

Tahap berikutnya: extract teks dari semua PDF, lalu lakukan cleaning + validasi.

In [ ]:
cleaner_mod = load_module('cleaner_mod', PROJECT_DIR / 'src' / '02_extract_and_clean.py')
results = cleaner_mod.process_all()
print(f'\nTotal diproses: {len(results)}')

## 5. Analisis Kualitas Hasil Cleaning

In [ ]:
with open('data/raw/processing_summary.json', 'r', encoding='utf-8') as f:
    summary = json.load(f)

df_sum = pd.DataFrame(summary)
print('Distribusi extraction status:')
print(df_sum['extraction_status'].value_counts())
print('\nDistribusi validation status:')
print(df_sum['validation_status'].value_counts())
print(f'\nStatistik word count (cleaned):')
print(df_sum['word_count_clean'].describe())

In [ ]:
valid_rows = df_sum[df_sum['validation_status'] == 'VALID']
if len(valid_rows) > 0:
    sample = valid_rows.iloc[0]
    with open(sample['cleaned_file'], 'r', encoding='utf-8') as f:
        text = f.read()
    print(f"Case: {sample['case_id']}  |  Word count: {sample['word_count_clean']}")
    print('--- PREVIEW (first 1500 chars) ---')
    print(text[:1500])

## 6. Visualisasi Statistik Case Base

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribusi word count
valid_df = df_sum[df_sum['extraction_status'] == 'OK']
axes[0].hist(valid_df['word_count_clean'], bins=15, color='steelblue', edgecolor='white')
axes[0].axvline(500, color='red', linestyle='--', label='Min 500 kata')
axes[0].set_title('Distribusi Word Count per Dokumen')
axes[0].set_xlabel('Jumlah kata')
axes[0].set_ylabel('Frekuensi')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Status validasi
val_counts = df_sum['validation_status'].value_counts()
colors = ['#2ecc71' if s == 'VALID' else '#e74c3c' if s == 'TOO_SHORT' else '#f39c12'
          for s in val_counts.index]
axes[1].bar(val_counts.index, val_counts.values, color=colors, edgecolor='black')
axes[1].set_title('Status Validasi per Dokumen')
axes[1].set_ylabel('Jumlah')
for i, v in enumerate(val_counts.values):
    axes[1].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

# Distribusi per tahun
tahun_counts = df_meta['tahun'].value_counts().sort_index()
axes[2].bar(tahun_counts.index.astype(str), tahun_counts.values,
            color='coral', edgecolor='black')
axes[2].set_title('Distribusi Putusan per Tahun')
axes[2].set_xlabel('Tahun')
axes[2].set_ylabel('Jumlah')

plt.tight_layout()
plt.savefig('reports/figures/case_base_stats.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Final Checklist

In [ ]:
valid_count = (df_sum['validation_status'] == 'VALID').sum()
total_count = len(df_sum)

print(f'Total dokumen diproses : {total_count}')
print(f'Dokumen VALID          : {valid_count}')
print(f'Pencapaian             : {valid_count}/{total_count} ({100*valid_count/total_count:.1f}%)')
print()

if valid_count >= 30:
    print('STATUS: SIAP lanjut ke Tahap 2 (Case Representation)')
else:
    print('STATUS: KURANG. Diperlukan minimal 30 dokumen VALID.')
    print('Tindakan: cek dokumen yang TOO_SHORT atau SUSPICIOUS,')
    print('ganti dengan putusan lain jika PDF hasil scan.')